# **Step 1: Install Required Libraries**

In [ ]:
!pip install scikit-learn pandas joblib

# **Step 2: Upload Your Dataset** (Upload the phishing_site_urls.csv)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving malicious_phish.csv to malicious_phish.csv


# **Step 2: Upload Your Dataset** (Upload the whois_data_real_100k.csv)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving whois_data_real_100k.csv to whois_data_real_100k.csv


## **Step 3: Load & Prepare the Dataset**

In [ ]:
import pandas as pd
import re

# Load phishing dataset
url_df = pd.read_csv("malicious_phish.csv")
url_df = url_df[['url', 'type']].dropna()
url_df['label'] = url_df['type'].apply(lambda x: 0 if x == 'benign' else 1)

# Extract domain from URLs
def extract_domain(url):
    match = re.findall(r'https?://([^/]+)', url)
    return match[0].lower() if match else 'unknown'

url_df['domain'] = url_df['url'].apply(extract_domain)

# Load WHOIS dataset
whois_df = pd.read_csv("whois_data_real_100k.csv")
whois_df['domain'] = whois_df['DOMAIN_NAME'].str.lower()
whois_df['domain_age'] = whois_df['DOMAIN_AGE']

# Merge
merged_df = pd.merge(url_df, whois_df[['domain', 'domain_age']], on='domain', how='left')
merged_df['domain_age'].fillna(merged_df['domain_age'].median(), inplace=True)

<ipython-input-4-cf62a6b243f2>:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['domain_age'].fillna(merged_df['domain_age'].median(), inplace=True)


# **Step 4: Feature Extraction**

In [ ]:
def extract_features(url):
    return {
        'url_length': len(url),
        'has_ip': 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0,
        'count_dots': url.count('.'),
        'count_hyphens': url.count('-'),
        'count_at': url.count('@'),
        'count_question': url.count('?'),
        'count_percent': url.count('%'),
        'count_equal': url.count('='),
        'https': 1 if url.startswith('https') else 0
    }

features_df = url_df['url'].apply(extract_features).apply(pd.Series)
features_df['domain_age'] = merged_df['domain_age']

X = features_df
y = merged_df['label']

# **Step 5: Train the Model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87     85778
           1       0.78      0.65      0.71     44461

    accuracy                           0.82    130239
   macro avg       0.81      0.78      0.79    130239
weighted avg       0.81      0.82      0.81    130239



# **STEP 6: Real-Time Prediction Function**

In [ ]:
def check_url(url):
    domain = extract_domain(url)
    age_row = whois_df[whois_df['domain'] == domain]
    age = int(age_row['domain_age'].values[0]) if not age_row.empty else int(whois_df['domain_age'].median())

    feats = extract_features(url)
    feats['domain_age'] = age
    features_df = pd.DataFrame([feats])
    proba = model.predict_proba(features_df)[0][1]
    risk = round(proba * 100, 2)

    reasons = []
    if feats['has_ip']: reasons.append("uses IP address")
    if feats['url_length'] > 75: reasons.append("very long URL")
    if feats['count_at'] > 0: reasons.append("contains '@'")
    if feats['https'] == 0: reasons.append("not using HTTPS")
    if feats['domain_age'] < 30: reasons.append("newly registered domain")
    if any(k in url.lower() for k in ["login", "verify", "update", "secure"]):
        reasons.append("contains phishing-related words")

    # ✅ NEW LOGIC: Only flag as MALICIOUS if risk is high AND multiple red flags
    if risk > 85 and len(reasons) > 1:
        prediction = 1
    else:
        prediction = 0

    reason_text = "; ".join(reasons) if reasons else "no suspicious pattern"

    print(f"\n🔎 URL: {url}")
    print(f"🛡️ Result: {'MALICIOUS ❌' if prediction else 'BENIGN ✅'}")
    print(f"📊 Risk Score: {risk}%")
    print(f"📌 Reason: {reason_text}")
    print(f"🌍 Domain Age: {age} days")


# **STEP 7: Run Demo Cases (STEP 8)**

In [ ]:
check_url("https://mail.google.com/mail/u/0/")


🔎 URL: https://mail.google.com/mail/u/0/
🛡️ Result: BENIGN ✅
📊 Risk Score: 100.0%
📌 Reason: no suspicious pattern
🌍 Domain Age: 9624 days


In [ ]:
print("====== SMART PHISHING DETECTOR DEMO (STEP 8) ======\n")

check_url("http://192.168.1.1/login")
check_url("http://apple-account23.com/update")
check_url("http://paypal-login7.com/verify")
check_url("http://secure-update10.com/login")
check_url("http://microsoftsupport-reset.com")
check_url("https://www.google.com")
check_url("https://github.com/login")
check_url("https://stackoverflow.com/questions")
check_url("https://www.wikipedia.org")
check_url("https://zoom.us/start")


====== SMART PHISHING DETECTOR DEMO (STEP 8) ======


🔎 URL: http://192.168.1.1/login
🛡️ Result: MALICIOUS ❌
📊 Risk Score: 96.0%
📌 Reason: uses IP address; not using HTTPS; contains phishing-related words
🌍 Domain Age: 3214 days

🔎 URL: http://apple-account23.com/update
🛡️ Result: MALICIOUS ❌
📊 Risk Score: 100.0%
📌 Reason: not using HTTPS; contains phishing-related words
🌍 Domain Age: 3214 days

🔎 URL: http://paypal-login7.com/verify
🛡️ Result: MALICIOUS ❌
📊 Risk Score: 100.0%
📌 Reason: not using HTTPS; contains phishing-related words
🌍 Domain Age: 3214 days

🔎 URL: http://secure-update10.com/login
🛡️ Result: MALICIOUS ❌
📊 Risk Score: 100.0%
📌 Reason: not using HTTPS; contains phishing-related words
🌍 Domain Age: 3214 days

🔎 URL: http://microsoftsupport-reset.com
🛡️ Result: BENIGN ✅
📊 Risk Score: 100.0%
📌 Reason: not using HTTPS
🌍 Domain Age: 3214 days

🔎 URL: https://www.google.com
🛡️ Result: BENIGN ✅
📊 Risk Score: 100.0%
📌 Reason: no suspicious pattern
🌍 Domain Age: 9623 days

🔎 URL